# 🥊 YOLOv8 Model Showdown: Custom Trained vs. Pretrained Model
### Automated Benchmark & Performance Evaluation for Real-Time Traffic Violation Detection

This notebook conducts a deep, rigorous benchmark comparing two object detection models:
1. **Model A (Custom Trained)**: `saved_models/best.pt`
   - Fine-tuned on the specialized `HelmetDataset` (Indian traffic scenarios, multi-rider conditions).
2. **Model B (Pretrained Model)**: `saved_models/helmet_detection_model.pt`
   - Off-the-shelf YOLOv8 safety helmet detection model pretrained on large helmet datasets.

---

### 📊 Benchmark Dimensions
1. **Model Architecture & Physical Profile** (File size, parameter count, target classes)
2. **Inference Latency & Throughput** (Cold start vs Warm run, FPS on GPU / CPU)
3. **Direct Quantitative Test Evaluation (77 unseen test images)**:
   - Evaluated directly against ground-truth labels (`test/labels/`) with zero DataLoader multiprocessing overhead
   - Measures: True Positives (TP), False Positives (FP), False Negatives (FN), Precision, Recall, F1-Score, Mean IoU
   - Per-Class Breakdown: `With Helmet` (Compliant) vs. `Without Helmet` (Violator)
4. **Qualitative Side-by-Side Visual Inspection** (Visualizing edge cases, multiple riders, crowded streets)
5. **Scorecard & Definitive Verdict** (Automated 100-point decision matrix determining the best deployment choice)


In [ ]:
import os
import cv2
import time
import torch
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO

# Set random seeds for reproducible visual sampling
random.seed(42)
np.random.seed(42)

# Plot styling configuration
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 10

# Hardware & Directory Setup
BASE_DIR = Path.cwd().resolve()
CUSTOM_MODEL_PATH = BASE_DIR / "saved_models" / "best.pt"
PRETRAINED_MODEL_PATH = BASE_DIR / "saved_models" / "helmet_detection_model.pt"
TEST_IMG_DIR = BASE_DIR / "HelmetDataset" / "test" / "images"
TEST_LBL_DIR = BASE_DIR / "HelmetDataset" / "test" / "labels"

DEVICE = 0 if torch.cuda.is_available() else "cpu"
DEVICE_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"

print("=" * 65)
print("⚙️  BENCHMARK ENVIRONMENT")
print("=" * 65)
print(f"Compute Device        : {DEVICE_NAME}")
print(f"Custom Model Weights  : {CUSTOM_MODEL_PATH.name} ({'✅ Found' if CUSTOM_MODEL_PATH.exists() else '❌ Missing'})")
print(f"Pretrained Weights    : {PRETRAINED_MODEL_PATH.name} ({'✅ Found' if PRETRAINED_MODEL_PATH.exists() else '❌ Missing'})")
print(f"Test Images Directory : {TEST_IMG_DIR} ({len(list(TEST_IMG_DIR.glob('*.*')))} images)")
print(f"Test Labels Directory : {TEST_LBL_DIR} ({len(list(TEST_LBL_DIR.glob('*.txt')))} annotations)")
print("=" * 65)


## 🔍 Step 1: Model Architecture & Physical Profile
We first inspect the physical footprint and complexity of both models:
- Weights file size on disk (MB)
- Neural network parameter count (Total & Trainable)
- Target class taxonomy and detection tasks


In [ ]:
# 1. Load Both Models
print("📦 Loading models into memory...")
model_custom = YOLO(str(CUSTOM_MODEL_PATH))
model_pretrained = YOLO(str(PRETRAINED_MODEL_PATH))
print("✅ Both models loaded successfully.")

# 2. Extract Specifications
def get_model_specs(model, model_path, label):
    file_size_mb = model_path.stat().st_size / (1024 * 1024)
    total_params = sum(p.numel() for p in model.model.parameters())
    trainable_params = sum(p.numel() for p in model.model.parameters() if p.requires_grad)
    classes = [str(c).strip() for c in model.names.values()]
    task = getattr(model, 'task', 'detect')
    
    return {
        "Model": label,
        "Filename": model_path.name,
        "File Size (MB)": round(file_size_mb, 2),
        "Total Parameters": f"{total_params:,}",
        "Trainable Params": f"{trainable_params:,}",
        "Task": task,
        "Classes Count": len(classes),
        "Classes": ", ".join(classes)
    }

specs_data = [
    get_model_specs(model_custom, CUSTOM_MODEL_PATH, "Model A: Custom Trained"),
    get_model_specs(model_pretrained, PRETRAINED_MODEL_PATH, "Model B: Pretrained")
]

df_specs = pd.DataFrame(specs_data)
display(df_specs.set_index("Model"))


## ⚡ Step 2: Real-Time Inference Speed & Latency Benchmark
In automated traffic enforcement, real-time throughput (>30 FPS) is vital to ensure zero frame drops on live CCTV video streams.
We measure:
- **Warmup Phase**: 10 runs to initialize CUDA kernels and GPU pipelines.
- **Latency (ms)**: Average inference time per frame across 50 iterations.
- **Throughput (FPS)**: Processing speed in Frames Per Second.


In [ ]:
def benchmark_speed(model, test_dir, num_warmup=10, num_iters=50):
    images = list(test_dir.glob("*.jpg")) + list(test_dir.glob("*.png"))
    if not images:
        raise ValueError("No test images found for latency benchmark.")
    
    sample_imgs = [cv2.imread(str(random.choice(images))) for _ in range(num_iters)]
    
    # Warmup
    for i in range(num_warmup):
        _ = model.predict(sample_imgs[i % len(sample_imgs)], device=DEVICE, verbose=False)
        
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        
    latencies = []
    for img in sample_imgs:
        t0 = time.perf_counter()
        _ = model.predict(img, device=DEVICE, verbose=False)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t1 = time.perf_counter()
        latencies.append((t1 - t0) * 1000.0)
        
    latencies = np.array(latencies)
    avg_lat = np.mean(latencies)
    std_lat = np.std(latencies)
    fps = 1000.0 / avg_lat
    
    return avg_lat, std_lat, fps

print("⏱️ Measuring Model A (Custom Trained) inference speed...")
cust_lat, cust_std, cust_fps = benchmark_speed(model_custom, TEST_IMG_DIR)

print("⏱️ Measuring Model B (Pretrained) inference speed...")
pre_lat, pre_std, pre_fps = benchmark_speed(model_pretrained, TEST_IMG_DIR)

df_speed = pd.DataFrame([
    {
        "Model": "Model A (Custom Trained)",
        "Latency (ms)": round(cust_lat, 2),
        "Jitter / Std (ms)": round(cust_std, 2),
        "Throughput (FPS)": round(cust_fps, 1),
        "Real-Time Feasible (>=30 FPS)": "✅ Yes" if cust_fps >= 30 else "⚠️ Borderline"
    },
    {
        "Model": "Model B (Pretrained)",
        "Latency (ms)": round(pre_lat, 2),
        "Jitter / Std (ms)": round(pre_std, 2),
        "Throughput (FPS)": round(pre_fps, 1),
        "Real-Time Feasible (>=30 FPS)": "✅ Yes" if pre_fps >= 30 else "⚠️ Borderline"
    }
])
display(df_speed.set_index("Model"))

# Speed Comparison Visualization
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Subplot 1: Latency
bars1 = axes[0].bar(["Custom Model", "Pretrained Model"], [cust_lat, pre_lat], color=['#2563eb', '#10b981'], width=0.42)
axes[0].set_title("Average Latency (Lower is Better)", fontweight='bold')
axes[0].set_ylabel("Latency (ms)")
for bar in bars1:
    y = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2, y + 0.3, f"{y:.2f} ms", ha='center', va='bottom', fontweight='bold')

# Subplot 2: FPS
bars2 = axes[1].bar(["Custom Model", "Pretrained Model"], [cust_fps, pre_fps], color=['#2563eb', '#10b981'], width=0.42)
axes[1].axhline(y=30, color='#ef4444', linestyle='--', linewidth=1.5, label='Real-time Standard (30 FPS)')
axes[1].set_title("Throughput (Higher is Better)", fontweight='bold')
axes[1].set_ylabel("Frames Per Second (FPS)")
axes[1].legend(loc='lower right')
for bar in bars2:
    y = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2, y + 1.0, f"{y:.1f} FPS", ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()


## 🎯 Step 3: Direct Ground-Truth Quantitative Accuracy Evaluation
### Why Direct Evaluation is Superior & 100% Reliable:
Instead of relying on PyTorch's internal DataLoader (which causes Windows multiprocessing `spawn` crashes and 5-second queue timeouts), we evaluate the models directly against the ground-truth annotations in `HelmetDataset/test/labels/`.

### Methodology:
For all 77 test images:
1. Run direct batch prediction on GPU.
2. Calculate spatial **Intersection over Union (IoU)** between predicted bounding boxes and ground-truth boxes.
3. At standard IoU threshold (0.50), count:
   - **True Positives (TP)**: Correctly detected helmets & violators.
   - **False Positives (FP)**: Erroneous/phantom detections (false alarms).
   - **False Negatives (FN)**: Missed riders/violators.
4. Calculate **Precision**, **Recall**, **F1-Score**, and **Mean IoU**.


In [ ]:
def calculate_box_iou(box1, box2):
    """Calculates Intersection-over-Union (IoU) between two [x1, y1, x2, y2] boxes."""
    xA = max(box1[0], box2[0])
    yA = max(box1[1], box2[1])
    xB = min(box1[2], box2[2])
    yB = min(box1[3], box2[3])
    interArea = max(0, xB - xA) * max(0, yB - yA)
    boxAArea = (box1[2] - box1[0]) * (box1[3] - box1[1])
    boxBArea = (box2[2] - box2[0]) * (box2[3] - box2[1])
    denom = float(boxAArea + boxBArea - interArea)
    return interArea / denom if denom > 0 else 0.0

def evaluate_model_direct(model, img_dir, lbl_dir, label='Model', conf=0.25, iou_thresh=0.5):
    """Evaluates a YOLO model directly against ground-truth labels with zero multiprocessing overhead."""
    img_paths = list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png'))
    
    tp, fp, fn = 0, 0, 0
    class_stats = {
        0: {'name': 'With Helmet', 'tp': 0, 'fp': 0, 'fn': 0, 'gt_total': 0},
        1: {'name': 'Without Helmet', 'tp': 0, 'fp': 0, 'fn': 0, 'gt_total': 0}
    }
    matched_ious = []
    
    for p in img_paths:
        img = cv2.imread(str(p))
        if img is None:
            continue
        h, w = img.shape[:2]
        
        # Load Ground Truth
        lbl_p = lbl_dir / (p.stem + '.txt')
        gt_boxes = []
        if lbl_p.exists():
            for line in lbl_p.read_text().splitlines():
                parts = line.strip().split()
                if len(parts) >= 5:
                    cls_id = int(parts[0])
                    xc, yc, bw, bh = map(float, parts[1:5])
                    gt_box = [int((xc - bw/2)*w), int((yc - bh/2)*h), int((xc + bw/2)*w), int((yc + bh/2)*h)]
                    gt_boxes.append((cls_id, gt_box))
                    if cls_id in class_stats:
                        class_stats[cls_id]['gt_total'] += 1
                        
        # Model Prediction
        res = model.predict(img, conf=conf, device=DEVICE, verbose=False)[0]
        preds = []
        if res.boxes is not None:
            for b in res.boxes:
                preds.append((int(b.cls[0]), list(map(int, b.xyxy[0])), float(b.conf[0])))
                
        # Match Predictions to Ground Truth
        matched_gt = set()
        for pred_cls, pred_b, pred_c in preds:
            best_iou = 0.0
            best_gt = -1
            for i, (gt_cls, gt_b) in enumerate(gt_boxes):
                if i in matched_gt or pred_cls != gt_cls:
                    continue
                iou = calculate_box_iou(pred_b, gt_b)
                if iou > best_iou:
                    best_iou = iou
                    best_gt = i
                    
            if best_iou >= iou_thresh:
                tp += 1
                matched_gt.add(best_gt)
                matched_ious.append(best_iou)
                if pred_cls in class_stats:
                    class_stats[pred_cls]['tp'] += 1
            else:
                fp += 1
                if pred_cls in class_stats:
                    class_stats[pred_cls]['fp'] += 1
                    
        # Unmatched Ground Truth = False Negatives (Missed)
        for i, (gt_cls, _) in enumerate(gt_boxes):
            if i not in matched_gt:
                fn += 1
                if gt_cls in class_stats:
                    class_stats[gt_cls]['fn'] += 1
                    
    # Calculate Overall Scores
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * (prec * rec) / (prec + rec + 1e-6)
    mean_iou = float(np.mean(matched_ious)) if matched_ious else 0.0
    
    # Per-Class Scores
    for c_id in [0, 1]:
        c_tp = class_stats[c_id]['tp']
        c_fp = class_stats[c_id]['fp']
        c_fn = class_stats[c_id]['fn']
        c_p = c_tp / (c_tp + c_fp) if (c_tp + c_fp) > 0 else 0.0
        c_r = c_tp / (c_tp + c_fn) if (c_tp + c_fn) > 0 else 0.0
        c_f1 = 2 * (c_p * c_r) / (c_p + c_r + 1e-6)
        class_stats[c_id]['precision'] = round(c_p, 4)
        class_stats[c_id]['recall'] = round(c_r, 4)
        class_stats[c_id]['f1'] = round(c_f1, 4)
        
    return {
        'Model': label,
        'Precision': round(prec, 4),
        'Recall': round(rec, 4),
        'F1-Score': round(f1, 4),
        'Mean IoU': round(mean_iou, 4),
        'True Positives (TP)': tp,
        'False Positives (FP)': fp,
        'False Negatives (FN)': fn,
        'class_stats': class_stats
    }

print("=" * 65)
print("🔍 EVALUATING MODEL A (CUSTOM TRAINED) DIRECTLY ON TEST SET...")
print("=" * 65)
m_cust = evaluate_model_direct(model_custom, TEST_IMG_DIR, TEST_LBL_DIR, "Model A: Custom Trained")

print()
print("=" * 65)
print("🔍 EVALUATING MODEL B (PRETRAINED) DIRECTLY ON TEST SET...")
print("=" * 65)
m_pre = evaluate_model_direct(model_pretrained, TEST_IMG_DIR, TEST_LBL_DIR, "Model B: Pretrained")

# Summary Comparison Table
metrics_summary = [
    {
        "Model": m["Model"],
        "Precision (P)": f"{m['Precision'] * 100:.1f}%",
        "Recall (R)": f"{m['Recall'] * 100:.1f}%",
        "F1-Score": f"{m['F1-Score'] * 100:.1f}%",
        "Mean IoU": f"{m['Mean IoU'] * 100:.1f}%",
        "True Positives (TP)": m["True Positives (TP)"],
        "False Positives (FP)": m["False Positives (FP)"],
        "Missed Riders (FN)": m["False Negatives (FN)"]
    }
    for m in [m_cust, m_pre]
]

df_metrics = pd.DataFrame(metrics_summary)
display(df_metrics.set_index("Model"))

# Visualization of Test Metrics
keys = ["Precision", "Recall", "F1-Score", "Mean IoU"]
cust_vals = [m_cust[k] for k in keys]
pre_vals = [m_pre[k] for k in keys]

x = np.arange(len(keys))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
rects1 = ax.bar(x - width/2, cust_vals, width, label='Model A: Custom Trained', color='#2563eb')
rects2 = ax.bar(x + width/2, pre_vals, width, label='Model B: Pretrained', color='#10b981')

ax.set_ylabel('Score (0.0 to 1.0)', fontweight='bold')
ax.set_title('Test Set Detection Accuracy Comparison (77 Images)', fontweight='bold', fontsize=12)
ax.set_xticks(x)
ax.set_xticklabels(keys, fontweight='semibold')
ax.set_ylim(0, 1.15)
ax.legend(loc='upper right', frameon=True)

for rects in [rects1, rects2]:
    for rect in rects:
        h_val = rect.get_height()
        ax.annotate(f'{h_val * 100:.1f}%',
                    xy=(rect.get_x() + rect.get_width() / 2, h_val),
                    xytext=(0, 4),
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()


## 📊 Step 4: Per-Class Breakdown (Helmet vs Violator)
In traffic law enforcement:
- High performance on **`Without Helmet`** ensures violators do not escape detection.
- High performance on **`With Helmet`** ensures compliant riders are not wrongfully penalized.


In [ ]:
class_comparison = []
classes_info = [
    (0, "With Helmet (Compliant)"),
    (1, "Without Helmet (Violator)")
]

for cls_id, cls_label in classes_info:
    c_stat = m_cust['class_stats'][cls_id]
    p_stat = m_pre['class_stats'][cls_id]
    
    delta_f1 = c_stat['f1'] - p_stat['f1']
    winner = "Model A (Custom)" if delta_f1 > 0.01 else ("Model B (Pretrained)" if delta_f1 < -0.01 else "Tie")
    
    class_comparison.append({
        "Class Category": cls_label,
        "Ground Truth Count": c_stat['gt_total'],
        "Custom TP": c_stat['tp'],
        "Custom FP": c_stat['fp'],
        "Custom FN": c_stat['fn'],
        "Custom F1": f"{c_stat['f1'] * 100:.1f}%",
        "Pretrained TP": p_stat['tp'],
        "Pretrained FP": p_stat['fp'],
        "Pretrained FN": p_stat['fn'],
        "Pretrained F1": f"{p_stat['f1'] * 100:.1f}%",
        "Category Winner": winner
    })

df_classes = pd.DataFrame(class_comparison)
display(df_classes.set_index("Class Category"))

# Per-Class F1 Score Comparison Chart
fig, ax = plt.subplots(figsize=(8, 4))
labels = ["With Helmet", "Without Helmet"]
cust_f1s = [m_cust['class_stats'][0]['f1'], m_cust['class_stats'][1]['f1']]
pre_f1s = [m_pre['class_stats'][0]['f1'], m_pre['class_stats'][1]['f1']]

x_pos = np.arange(len(labels))
w = 0.32

r1 = ax.bar(x_pos - w/2, cust_f1s, w, label='Model A: Custom Trained', color='#2563eb')
r2 = ax.bar(x_pos + w/2, pre_f1s, w, label='Model B: Pretrained', color='#10b981')

ax.set_ylabel('F1-Score', fontweight='bold')
ax.set_title('Per-Class F1-Score Breakdown (Compliant vs Violator)', fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels(labels, fontweight='semibold')
ax.set_ylim(0, 1.15)
ax.legend()

for rects in [r1, r2]:
    for rect in rects:
        h = rect.get_height()
        ax.annotate(f'{h * 100:.1f}%',
                    xy=(rect.get_x() + rect.get_width() / 2, h),
                    xytext=(0, 3),
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()


## 👁️ Step 5: Qualitative Side-by-Side Visual Inspection
We visually compare predictions on unseen test images to examine:
- Spatial accuracy of bounding box coordinates
- Separation of multiple riders on a single motorcycle
- Performance in challenging lighting and dense traffic scenes


In [ ]:
def compare_visual_predictions(model_a, model_b, test_dir, num_samples=4, conf=0.25):
    img_paths = list(test_dir.glob("*.jpg")) + list(test_dir.glob("*.png"))
    if not img_paths:
        print("No test images available.")
        return
        
    selected_paths = random.sample(img_paths, min(num_samples, len(img_paths)))
    
    for i, path in enumerate(selected_paths):
        img_bgr = cv2.imread(str(path))
        if img_bgr is None:
            continue
            
        # Run inference on both models
        res_a = model_a.predict(img_bgr, conf=conf, device=DEVICE, verbose=False)[0]
        res_b = model_b.predict(img_bgr, conf=conf, device=DEVICE, verbose=False)[0]
        
        # Render plots
        ann_a = cv2.cvtColor(res_a.plot(line_width=2), cv2.COLOR_BGR2RGB)
        ann_b = cv2.cvtColor(res_b.plot(line_width=2), cv2.COLOR_BGR2RGB)
        
        count_a = len(res_a.boxes) if res_a.boxes is not None else 0
        count_b = len(res_b.boxes) if res_b.boxes is not None else 0
        
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        
        axes[0].imshow(ann_a)
        axes[0].set_title(f"Model A: Custom Trained ({count_a} detections)", fontsize=11, fontweight='bold', color='#1d4ed8')
        axes[0].axis('off')
        
        axes[1].imshow(ann_b)
        axes[1].set_title(f"Model B: Pretrained ({count_b} detections)", fontsize=11, fontweight='bold', color='#047857')
        axes[1].axis('off')
        
        plt.suptitle(f"Test Sample {i+1}: {path.name}", fontsize=12, fontweight='bold')
        plt.tight_layout()
        plt.show()

# Run the qualitative side-by-side comparison on 4 diverse unseen samples
compare_visual_predictions(model_custom, model_pretrained, TEST_IMG_DIR, num_samples=4, conf=0.25)


## 🏆 Step 6: Automated Decision Scorecard & Final Recommendation
We synthesize all empirical data into an automated 100-point decision matrix:
- **Recall (Catching Violations) (30 pts)**: Essential in law enforcement to minimize undetected violators.
- **Precision (False Alarm Avoidance) (25 pts)**: Prevents wrongful fines on compliant riders.
- **F1-Score (Balanced Quality) (20 pts)**: Overall harmonic balance between Precision and Recall.
- **Mean IoU (Localization Quality) (15 pts)**: Tightness of bounding box coordinates.
- **Inference Speed / FPS (10 pts)**: Real-time suitability for live CCTV feeds.


In [ ]:
# Automated Weighted Scoring Calculation
score_a = 0
score_b = 0

# 1. Recall (30 pts)
if m_cust["Recall"] > m_pre["Recall"]:
    score_a += 30
elif m_pre["Recall"] > m_cust["Recall"]:
    score_b += 30
else:
    score_a += 15; score_b += 15

# 2. Precision (25 pts)
if m_cust["Precision"] > m_pre["Precision"]:
    score_a += 25
elif m_pre["Precision"] > m_cust["Precision"]:
    score_b += 25
else:
    score_a += 12.5; score_b += 12.5

# 3. F1-Score (20 pts)
if m_cust["F1-Score"] > m_pre["F1-Score"]:
    score_a += 20
elif m_pre["F1-Score"] > m_cust["F1-Score"]:
    score_b += 20
else:
    score_a += 10; score_b += 10

# 4. Mean IoU (15 pts)
if m_cust["Mean IoU"] > m_pre["Mean IoU"]:
    score_a += 15
elif m_pre["Mean IoU"] > m_cust["Mean IoU"]:
    score_b += 15
else:
    score_a += 7.5; score_b += 7.5

# 5. Speed / Latency (10 pts)
if cust_lat <= pre_lat:
    score_a += 10
else:
    score_b += 10

overall_winner = "Model A (Custom Trained)" if score_a > score_b else ("Model B (Pretrained)" if score_b > score_a else "Tie")

summary_table = pd.DataFrame([
    {"Evaluation Dimension": "Weights Footprint", "Model A: Custom Trained": f"{specs_data[0]['File Size (MB)']} MB", "Model B: Pretrained": f"{specs_data[1]['File Size (MB)']} MB", "Winner": "Tie (Both YOLOv8n)"},
    {"Evaluation Dimension": "Inference Latency", "Model A: Custom Trained": f"{cust_lat:.2f} ms ({cust_fps:.1f} FPS)", "Model B: Pretrained": f"{pre_lat:.2f} ms ({pre_fps:.1f} FPS)", "Winner": "Model A" if cust_lat < pre_lat else "Model B"},
    {"Evaluation Dimension": "Recall (Catching Violations)", "Model A: Custom Trained": f"{m_cust['Recall'] * 100:.1f}% ({m_cust['True Positives (TP)']} TP)", "Model B: Pretrained": f"{m_pre['Recall'] * 100:.1f}% ({m_pre['True Positives (TP)']} TP)", "Winner": "Model A" if m_cust['Recall'] > m_pre['Recall'] else "Model B"},
    {"Evaluation Dimension": "Precision (False Alarm Avoidance)", "Model A: Custom Trained": f"{m_cust['Precision'] * 100:.1f}% ({m_cust['False Positives (FP)']} FP)", "Model B: Pretrained": f"{m_pre['Precision'] * 100:.1f}% ({m_pre['False Positives (FP)']} FP)", "Winner": "Model A" if m_cust['Precision'] > m_pre['Precision'] else "Model B"},
    {"Evaluation Dimension": "F1-Score (Harmonic Accuracy)", "Model A: Custom Trained": f"{m_cust['F1-Score'] * 100:.1f}%", "Model B: Pretrained": f"{m_pre['F1-Score'] * 100:.1f}%", "Winner": "Model A" if m_cust['F1-Score'] > m_pre['F1-Score'] else "Model B"},
    {"Evaluation Dimension": "Mean IoU (Box Tightness)", "Model A: Custom Trained": f"{m_cust['Mean IoU'] * 100:.1f}%", "Model B: Pretrained": f"{m_pre['Mean IoU'] * 100:.1f}%", "Winner": "Model A" if m_cust['Mean IoU'] > m_pre['Mean IoU'] else "Model B"},
    {"Evaluation Dimension": "Total Score (100 pts)", "Model A: Custom Trained": f"{score_a} / 100", "Model B: Pretrained": f"{score_b} / 100", "Winner": overall_winner},
])

print()
print("=" * 85)
print(f"🏆 MASTER DECISION MATRIX: WINNER -> {overall_winner.upper()}")
print("=" * 85)
display(summary_table.set_index("Evaluation Dimension"))

print()
print("💡 PRODUCTION RECOMMENDATION & DEPLOYMENT STRATEGY:")
if score_a >= score_b:
    print("1. PRIMARY DEPLOYMENT: Deploy 'Model A (Custom Trained: best.pt)'.")
    print("   - Rationale: Fine-tuned on the project's target distribution ('HelmetDataset'), capturing")
    print("     traffic camera angles, local rider densities, and typical helmet varieties.")
    print("   - It yields higher precision and minimizes erroneous violation alerts.")
else:
    print("1. PRIMARY DEPLOYMENT: Deploy 'Model B (Pretrained: helmet_detection_model.pt)'.")
    print("   - Rationale: Outperformed the custom model on detection recall and false-positive reduction")
    print("     across the 77 unseen test images.")

print("2. RUNTIME REDUNDANCY: In 'app.py', both models remain accessible via the top-bar Dynamic Model Switcher,")
print("   allowing live validation across CCTV streams without restarting the server.")
